# MTPL2 Frequency Modelling Walkthrough

End-to-end Poisson frequency model on the French MTPL2 dataset (678k
policies). Covers data prep, model fitting, relativity inspection,
and the impact of discretising smooth curves into rating tables.

Sections:

1. **Data loading & preparation**
2. **Model specification** — splines, categoricals, numerics
3. **Fitting with REML** — automatic smoothness selection
4. **Relativity plots** — grid, individual terms, layout options
5. **Summary table** — statsmodels-style output
6. **Discretisation impact** — binning smooth curves into rating tables
7. **Rating tables** — per-feature step functions for deployment

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from superglm import Categorical, Numeric, Poisson, Spline, SuperGLM

## 1. Data loading & preparation

The French MTPL2 frequency dataset contains 678k motor third-party
liability policies with claim counts and exposure. We fetch it from
OpenML (no local files needed) and apply standard actuarial cleaning.

In [ ]:
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame

# Standard cleaning
df["ClaimNb"] = df["ClaimNb"].astype(float).clip(upper=4)
df["Exposure"] = df["Exposure"].astype(float).clip(lower=0.01)
df["DrivAge"] = df["DrivAge"].astype(float).clip(18, 90)
df["VehAge"] = df["VehAge"].astype(float).clip(0, 20)
df["BonusMalus"] = df["BonusMalus"].astype(float).clip(50, 150)
df["Density"] = df["Density"].astype(float)
df["LogDensity"] = np.log1p(df["Density"])

y = df["ClaimNb"].values
exposure = df["Exposure"].values

features_to_use = ["DrivAge", "VehAge", "BonusMalus", "LogDensity", "Area", "VehGas"]
X = df[features_to_use]

print(f"Rows: {len(df):,}")
print(f"Claim rate: {y.sum() / exposure.sum():.4f}")
print(f"Zero fraction: {(y == 0).mean():.1%}")
X.describe().round(2)

## 2. Model specification

A typical actuarial frequency model mixes feature types:

| Feature | Type | Why |
|---------|------|-----|
| DrivAge | `Spline(kind="bs", n_knots=15)` | Smooth U-shaped effect |
| VehAge | `Spline(kind="bs", n_knots=10)` | Smooth decreasing effect |
| BonusMalus | `Spline(kind="bs", n_knots=8)` | Strongly monotone, needs flexibility |
| LogDensity | `Numeric()` | Log-linear in population density |
| Area | `Categorical()` | 6 unordered regions |
| VehGas | `Categorical()` | Diesel vs Regular |

In [ ]:
model = SuperGLM(
    family=Poisson(),
    discrete=True,
    features={
        "DrivAge": Spline(kind="cr", n_knots=15, knot_strategy="quantile_rows", penalty="ssp"),
        "VehAge": Spline(kind="cr", n_knots=10, knot_strategy="uniform", penalty="ssp"),
        "BonusMalus": Spline(
            kind="cr", n_knots=8, knot_strategy="quantile_tempered", knot_alpha=0.2, penalty="ssp"
        ),
        "LogDensity": Numeric(),
        "Area": Categorical(base="most_exposed"),
        "VehGas": Categorical(base="most_exposed"),
    },
)
print("Model specified.")

## 3. Fitting with REML

`fit_reml()` estimates smoothing parameters automatically via
restricted marginal likelihood (Wood 2011). No manual tuning of
lambda needed — REML finds the right amount of smoothing for
each spline.

In [ ]:
t0 = time.perf_counter()
model.fit_reml(X, y, exposure=exposure)
elapsed = time.perf_counter() - t0

print(f"Fit time: {elapsed:.1f}s")
print(f"REML converged: {model._reml_result.converged}")
print(f"REML iterations: {model._reml_result.n_reml_iter}")
print("\nPer-feature edf:")
for name in model._feature_order:
    ti = model.term_inference(name, with_se=False)
    edf_str = f"{ti.edf:.1f}" if ti.edf is not None else "—"
    print(f"  {name:15s} edf = {edf_str}")

## 4. Relativity plots

### 4a. All features in a grid

The default `model.plot()` shows all main effects in a 2-column grid
with exposure density overlays and pointwise CIs.

In [ ]:
fig = model.plot(X=X, sample_weight=exposure)
plt.show()

### 4b. Single term — large format

Plot one term at a time for detailed inspection. Useful for
presentations or reports where you want to focus on one effect.

In [ ]:
fig = model.plot(
    "DrivAge",
    X=X,
    sample_weight=exposure,
    show_knots=True,
    figsize=(8, 6),
    title="Driver Age Effect",
    subtitle="Poisson frequency model, REML smoothing",
)
plt.show()

### 4c. Subset of terms

Pass a list of feature names to plot a subset in a grid.

In [ ]:
fig = model.plot(
    ["DrivAge", "VehAge", "BonusMalus"],
    X=X,
    sample_weight=exposure,
    ncols=3,
    figsize=(16, 5),
    title="Smooth terms",
)
plt.show()

### 4d. Simultaneous confidence bands

Pointwise CIs are narrow but don't account for multiple testing across
the curve. Simultaneous bands (Wood 2006, posterior simulation) give
honest coverage for the entire curve.

In [ ]:
fig = model.plot(
    "DrivAge",
    ci="both",
    X=X,
    sample_weight=exposure,
    figsize=(10, 5),
    title="Pointwise vs Simultaneous Bands",
)
plt.show()

## 5. Summary table

statsmodels-style summary with coefficient estimates, smooth term
edf, and Wood (2013) p-values for smooth terms.

In [ ]:
model.summary()

## 6. Discretisation impact

In production, smooth spline curves are typically discretised into
step-function rating tables (e.g. 10–20 age bands). The question is:
how many bins do you need before the approximation stops mattering?

`discretization_impact()` replaces each smooth per-observation
log-relativity with an exposure-weighted bin average, recomputes
predictions, and reports how much the deviance and predictions shift.

In [ ]:
# Compare bin counts: how many bins before the error is negligible?
for n_bins in [5, 10, 20, 50, 100]:
    r = model.discretization_impact(X, y, exposure=exposure, n_bins=n_bins)
    m = r.metrics
    print(
        f"  {n_bins:3d} bins: "
        f"deviance change {m['deviance_change_pct']:+.4f}%, "
        f"mean pred change {m['mean_abs_prediction_change_pct']:.4f}%, "
        f"corr {m['prediction_correlation']:.6f}"
    )

## 7. Rating tables

With a practical bin count chosen, extract the per-feature rating
tables and overlay the step function on the smooth curve.

In [ ]:
result = model.discretization_impact(X, y, exposure=exposure, n_bins=10)

spline_names = ["DrivAge", "VehAge", "BonusMalus"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, name in zip(axes, spline_names):
    # Smooth curve
    ti = model.term_inference(name)
    ax.plot(ti.x, ti.relativity, color="#1f77b4", linewidth=2, label="Smooth")

    # Step function from rating table
    table = result.tables[name]
    step_x, step_y = [], []
    for _, row in table.iterrows():
        step_x.extend([row["bin_from"], row["bin_to"]])
        step_y.extend([row["relativity"], row["relativity"]])
    ax.plot(step_x, step_y, color="#d62728", linewidth=1.5, label=f"Binned ({len(table)} bins)")

    ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title(name, fontweight="bold")
    ax.set_ylabel("Relativity")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.2, axis="y")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Smooth curves vs 10-bin rating tables", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Inspect one rating table
print("DrivAge rating table (10 bins, exposure-quantile):\n")
result.tables["DrivAge"]

## 8. Key takeaways

| | Detail |
|---|---|
| **REML** | Automatic smoothness — no manual lambda tuning |
| **Discretisation** | 10–20 bins per feature is enough for < 0.01% deviance change |
| **Rating tables** | `discretization_impact()` gives deployment-ready step functions |
| **Workflow** | Fit smooth → inspect curves → discretise for production |

The smooth model is the statistical model. The rating table is
what goes into the pricing engine. `discretization_impact()` bridges
the two and quantifies the approximation error.